# DAML Project 4 — Exotic searches with ATLAS and ML classification

### Student Name: Saqib Umair

## Table of Content
1. Introduction
2. Setup
3. Fitting and hypothesis testing
4. NN without the reco_zv_mass
5. NN including the reco_zv_mass
6. Comparison of Section 5 and Section 6
7. Conclusion 
8. Acknowledgement 

## 1. Introduction
In this notebook, I study a weighted search for a heavy resonance in the dilepton-plus-fatjet mass spectrum. The main idea is to see whether there is evidence of a signal around 1 TeV and to make that possible signal stand out more clearly from the background.

I start with a cut-based analysis, where I apply simple kinematic cuts to reduce the background in a controlled and understandable way. After that, I look at the mass spectrum, fit the signal and background shapes, and compare the background-only and signal-plus-background cases using Wilks’ theorem.

Then I move to the neural-network part of the analysis to check whether a classifier can separate signal and background better than the cut-based method. I first train the network without using `reco_zv_mass`, because I want to avoid directly biasing the variable that is later used for the final mass-spectrum test. In the last part, I train a second network that includes `reco_zv_mass` so I can see how much the classifier performance changes and whether the mass distribution becomes more sculpted.

So overall, this notebook follows the project step by step: first the cut-based study, then the hypothesis test, then a neural network without mass, and finally a neural network with mass included.

> **Public portfolio notebook:** The full original coursework source has been restored. Saved outputs are cleared and machine-specific paths replaced. Input course datasets are not redistributed; reproducing the analysis requires permitted files described in `data/README.md`. Reported results are historical project outputs, not new independent executions.

### Important libraries

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, roc_curve, roc_auc_score, classification_report
from sklearn.neural_network import MLPClassifier
from sklearn.utils import shuffle

from scipy.optimize import minimize
from scipy.stats import chi2, norm

# 2. Setup
## 2.1 Data Loading
I start by loading the signal sample and the three background samples: Diboson, Z+jets, and Top. Before doing the analysis, I check the main variables I use in the study for missing values, and in this case no extra cleaning is needed.

For plotting and fitting, I convert the mass-type variables from MeV to GeV because the numbers are much easier to read that way in the figures. I do not change the event weights, since those must stay the same throughout the analysis. When needed, I also merge the three background samples into a single combined background sample.

One important point here is that the event yield is not simply the number of rows in the dataframe. In this project, the meaningful yield comes from the sum of `FullEventWeight`, because these weights are what set the correct relative contribution of each process.

In [ ]:
# Here I set the folder where all the project files are saved
data_folder = Path("../data/raw")
if not data_folder.is_dir():
    data_folder = Path("data/raw")

# Load the signal sample and the three background samples
signal_df = pd.read_csv(data_folder / "ggH1000.csv")
diboson_df = pd.read_csv(data_folder / "Diboson.csv")
zjets_df = pd.read_csv(data_folder / "Zjets.csv")
top_df = pd.read_csv(data_folder / "Top.csv")

# Put all samples together so I can check them in the same way
samples = {
    "signal": signal_df,
    "diboson": diboson_df,
    "zjets": zjets_df,
    "top": top_df,
}

# These labels are just for cleaner printing
sample_labels = {
    "signal": "Signal (ggH1000)",
    "diboson": "Diboson",
    "zjets": "Z+jets",
    "top": "Top",
}

# First I print a small summary to make sure the files loaded correctly
print("Files loaded.")
for name, df in samples.items():
    print(f"{sample_labels[name]}: {len(df):,} rows")

# These are the columns I will use in the analysis
required_columns = [
    "lep1_pt",
    "lep2_pt",
    "fatjet_pt",
    "fatjet_eta",
    "fatjet_D2",
    "Zll_mass",
    "Zll_pt",
    "MET",
    "reco_zv_mass",
    "isSignal",
    "FullEventWeight",
]

# Check that all needed columns are present in every sample
for name, df in samples.items():
    missing_columns = [col for col in required_columns if col not in df.columns]
    if missing_columns:
        raise ValueError(f"{sample_labels[name]} is missing: {missing_columns}")

# Check missing values only in the variables that matter for this study
print("\nMissing values in the columns used here:")
for name, df in samples.items():
    missing_total = df[required_columns].isna().sum().sum()
    print(f"{sample_labels[name]}: {missing_total:,}")

# Clean the samples in a simple way
# Replace inf values with NaN, then drop rows missing any required variable
for name in samples:
    samples[name] = (
        samples[name]
        .replace([np.inf, -np.inf], np.nan)
        .dropna(subset=required_columns)
        .copy()
    )

# Save the cleaned versions back into separate dataframes
signal_df = samples["signal"]
diboson_df = samples["diboson"]
zjets_df = samples["zjets"]
top_df = samples["top"]

# Print the number of rows after cleaning
print("\nRows after cleaning:")
for name, df in samples.items():
    print(f"{sample_labels[name]}: {len(df):,}")


## 2.2 Kinematic distributions and choice of cuts
I first look at the kinematic distributions of the main input variables to see where the signal and background are separated most clearly. Since the project specifically asks for simple square cuts chosen by eye, I use these overlaid plots to guide the selection instead of doing a full optimisation.

I tested several cut values before choosing the final ones. The aim was not simply to make the biggest possible signal bump appear, but to find a sensible balance. The cuts should reduce the background and make any signal more visible, while still keeping a mass spectrum that looks smooth enough to fit properly afterwards.

Because of that, I do not apply any direct cut on `reco_zv_mass`, and I also avoid making the cuts too aggressive. This is important because very hard cuts can sculpt the mass distribution, which would make the background shape less trustworthy in the later fit. That is also in line with the warning given in the project brief.

In [ ]:

# Create GeV versions of the main kinematic variables
# The original ntuples store these quantities in MeV.
# For plotting and printed results, GeV is more readable,
# so I create extra columns with a "_gev" suffix.
variables_in_gev = [
    "lep1_pt",
    "lep2_pt",
    "fatjet_pt",
    "Zll_mass",
    "Zll_pt",
    "MET",
    "reco_zv_mass",
]

# Add GeV versions to each dataframe without changing the original values
for df in [signal_df, diboson_df, zjets_df, top_df]:
    for variable_name in variables_in_gev:
        df[f"{variable_name}_gev"] = df[variable_name] / 1000.0

In [ ]:
# Compare the main kinematic distributions for all processes

# These variables are plotted in GeV where appropriate so the axes are easier to read.
# I use normalised histograms here because I want to compare shapes, not absolute yields.
plot_variables = [
    "lep1_pt_gev",
    "lep2_pt_gev",
    "fatjet_pt_gev",
    "fatjet_eta",
    "fatjet_D2",
    "Zll_mass_gev",
    "Zll_pt_gev",
    "MET_gev",
    "reco_zv_mass_gev",
]

# Axis labels for cleaner figure presentation
axis_labels = {
    "lep1_pt_gev": r"Leading lepton $p_T$ [GeV]",
    "lep2_pt_gev": r"Subleading lepton $p_T$ [GeV]",
    "fatjet_pt_gev": r"Fat-jet $p_T$ [GeV]",
    "fatjet_eta": r"Fat-jet $\eta$",
    "fatjet_D2": r"Fat-jet $D_2$",
    "Zll_mass_gev": r"$m_{\ell\ell}$ [GeV]",
    "Zll_pt_gev": r"$p_T(\ell\ell)$ [GeV]",
    "MET_gev": r"MET [GeV]",
    "reco_zv_mass_gev": r"$m_{\ell\ell J}$ [GeV]",
}

# Set up a grid of subplots
plots_per_row = 3
number_of_rows = int(np.ceil(len(plot_variables) / plots_per_row))

fig, axes = plt.subplots(number_of_rows, plots_per_row, figsize=(18, 5 * number_of_rows))
axes = axes.flatten()

for i, variable_name in enumerate(plot_variables):
    ax = axes[i]

    # Build a combined array so I can define a sensible plotting range
    # that excludes extreme outliers from all samples together.
    all_values = np.concatenate([
        signal_df[variable_name].to_numpy(),
        diboson_df[variable_name].to_numpy(),
        zjets_df[variable_name].to_numpy(),
        top_df[variable_name].to_numpy(),
    ])

    x_min = np.percentile(all_values, 0.5)
    x_max = np.percentile(all_values, 99.5)

    # Physical lower limits should not go below zero for these variables
    if variable_name in [
        "lep1_pt_gev",
        "lep2_pt_gev",
        "fatjet_pt_gev",
        "fatjet_D2",
        "Zll_mass_gev",
        "Zll_pt_gev",
        "MET_gev",
        "reco_zv_mass_gev",
    ]:
        x_min = max(0, x_min)

    # Draw one normalised histogram for each process
    for name, df in samples.items():
        values = df[variable_name]
        values = values[(values >= x_min) & (values <= x_max)]

        ax.hist(
            values,
            bins=50,
            density=True,
            histtype="step",
            linewidth=1.7,
            label=sample_labels[name],
        )

    ax.set_xlabel(axis_labels[variable_name])
    ax.set_ylabel("Normalised entries")
    ax.set_title(axis_labels[variable_name])
    ax.legend(fontsize=8)

# Remove any unused subplot panels
for j in range(len(plot_variables), len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

In [ ]:

# I use this when I want to compare the signal against the total background
background_df = pd.concat([diboson_df, zjets_df, top_df], ignore_index=True)

# These are the variables I want to inspect before choosing the cuts
cut_variables = [
    "lep1_pt_gev",
    "lep2_pt_gev",
    "fatjet_pt_gev",
    "fatjet_D2",
    "Zll_pt_gev",
    "MET_gev",
]

# Make one figure with all variable distributions together
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, variable_name in enumerate(cut_variables):
    ax = axes[i]

    # Take the signal and background values for this variable
    signal_values = signal_df[variable_name].to_numpy()
    background_values = background_df[variable_name].to_numpy()

    # Use a trimmed plotting range so extreme outliers do not stretch the axis too much
    combined_values = np.concatenate([signal_values, background_values])
    x_min = np.percentile(combined_values, 0.5)
    x_max = np.percentile(combined_values, 99.5)

    # For these variables, negative values are not physically meaningful,
    # so I do not let the lower edge go below zero
    if variable_name in [
        "lep1_pt_gev",
        "lep2_pt_gev",
        "fatjet_pt_gev",
        "fatjet_D2",
        "Zll_pt_gev",
        "MET_gev",
    ]:
        x_min = max(0, x_min)

    # Keep only the values inside the chosen plotting range
    signal_values = signal_values[(signal_values >= x_min) & (signal_values <= x_max)]
    background_values = background_values[(background_values >= x_min) & (background_values <= x_max)]

    # Plot the signal shape
    ax.hist(
        signal_values,
        bins=50,
        density=True,
        histtype="step",
        linewidth=2.0,
        label="Signal",
    )

    # Plot the total background shape on the same axis
    ax.hist(
        background_values,
        bins=50,
        density=True,
        histtype="step",
        linewidth=2.0,
        label="Combined background",
    )

    # Add labels and title so the plots are easy to read
    ax.set_xlabel(axis_labels[variable_name])
    ax.set_ylabel("Normalised entries")
    ax.set_title(axis_labels[variable_name])
    ax.legend()

plt.tight_layout()
plt.show()

## 2.3 Weighted Yield and Applying the cuts
 In this part, i will apply cut to the datasets and calculate the yeild before and after the cut and than I will make its summary.

In [ ]:
# Baseline cuts

# I define one function for the cut-based selection so I can apply the same cuts everywhere
def apply_cuts(df):
    return (
        (df["lep1_pt"] > 135000) &
        (df["lep2_pt"] > 50000) &
        (df["fatjet_pt"] > 260000) &
        (np.abs(df["fatjet_eta"]) < 2.3) &
        (df["fatjet_D2"] < 2.2) &
        (df["Zll_mass"] > 78000) &
        (df["Zll_mass"] < 107000) &
        (df["Zll_pt"] > 150000) &
        (df["MET"] < 100000)
    )

# Keep a copy of the original samples before any selection
signal_before_cuts = signal_df.copy()
diboson_before_cuts = diboson_df.copy()
zjets_before_cuts = zjets_df.copy()
top_before_cuts = top_df.copy()

# Apply the baseline cuts to each sample separately
signal_after_cuts = signal_before_cuts[apply_cuts(signal_before_cuts)].copy()
diboson_after_cuts = diboson_before_cuts[apply_cuts(diboson_before_cuts)].copy()
zjets_after_cuts = zjets_before_cuts[apply_cuts(zjets_before_cuts)].copy()
top_after_cuts = top_before_cuts[apply_cuts(top_before_cuts)].copy()

# Combine the three background samples before cuts
background_before_cuts = pd.concat(
    [diboson_before_cuts, zjets_before_cuts, top_before_cuts],
    ignore_index=True
)

# Combine the three background samples again after cuts
background_after_cuts = pd.concat(
    [diboson_after_cuts, zjets_after_cuts, top_after_cuts],
    ignore_index=True
)

# Convert the reconstructed mass from MeV to GeV for easier plotting later
for df in [
    signal_before_cuts,
    diboson_before_cuts,
    zjets_before_cuts,
    top_before_cuts,
    signal_after_cuts,
    diboson_after_cuts,
    zjets_after_cuts,
    top_after_cuts,
    background_before_cuts,
    background_after_cuts,
]:
    df["reco_zv_mass_gev"] = df["reco_zv_mass"] / 1000.0

# In this project, the important yield is the weighted yield, not just the row count
def weighted_yield(df):
    return df["FullEventWeight"].sum()

# Calculate weighted yields before cuts
signal_yield_before = weighted_yield(signal_before_cuts)
diboson_yield_before = weighted_yield(diboson_before_cuts)
zjets_yield_before = weighted_yield(zjets_before_cuts)
top_yield_before = weighted_yield(top_before_cuts)
background_yield_before = weighted_yield(background_before_cuts)

# Calculate weighted yields after cuts
signal_yield_after = weighted_yield(signal_after_cuts)
diboson_yield_after = weighted_yield(diboson_after_cuts)
zjets_yield_after = weighted_yield(zjets_after_cuts)
top_yield_after = weighted_yield(top_after_cuts)
background_yield_after = weighted_yield(background_after_cuts)

# Print the weighted event yields before selection
print("\nBefore cuts")
print(f"Signal: {signal_yield_before:,.3f}")
print(f"Diboson: {diboson_yield_before:,.3f}")
print(f"Z+jets: {zjets_yield_before:,.3f}")
print(f"Top: {top_yield_before:,.3f}")
print(f"Total background: {background_yield_before:,.3f}")

# Print the weighted event yields after selection
print("\nAfter cuts")
print(f"Signal: {signal_yield_after:,.3f}")
print(f"Diboson: {diboson_yield_after:,.3f}")
print(f"Z+jets: {zjets_yield_after:,.3f}")
print(f"Top: {top_yield_after:,.3f}")
print(f"Total background: {background_yield_after:,.3f}")

# Work out how much signal and background survive the cuts
signal_efficiency = signal_yield_after / signal_yield_before
background_efficiency = background_yield_after / background_yield_before

# Also compare simple sensitivity numbers before and after the cuts
signal_to_background_before = signal_yield_before / background_yield_before
signal_to_background_after = signal_yield_after / background_yield_after

signal_over_sqrt_background_before = signal_yield_before / np.sqrt(background_yield_before)
signal_over_sqrt_background_after = signal_yield_after / np.sqrt(background_yield_after)

# Print a short cut summary
print("\nCut summary")
print(f"Signal efficiency: {signal_efficiency:.4f}")
print(f"Background efficiency: {background_efficiency:.4f}")
print(f"S/B before: {signal_to_background_before:.6f}")
print(f"S/B after: {signal_to_background_after:.6f}")
print(f"S/sqrt(B) before: {signal_over_sqrt_background_before:.6f}")
print(f"S/sqrt(B) after: {signal_over_sqrt_background_after:.6f}")

# Put the main weighted yields into a small summary table
selection_summary = pd.DataFrame(
    {
        "before": [
            signal_yield_before,
            diboson_yield_before,
            zjets_yield_before,
            top_yield_before,
            background_yield_before,
        ],
        "after": [
            signal_yield_after,
            diboson_yield_after,
            zjets_yield_after,
            top_yield_after,
            background_yield_after,
        ],
    },
    index=["Signal", "Diboson", "Z+jets", "Top", "Total background"],
)

selection_summary


To measure how much the cuts help, I compare the weighted signal and background yields before and after the selection. Before applying any cuts, the weighted signal yield is 4200.16, while the total weighted background is about 5.40 million. After the cuts, the signal yield drops to 2901.36 and the background falls much more strongly to 511349.75.

This means the signal efficiency is 0.6908, so around 69% of the signal is kept, while the background efficiency is 0.0946, which means only about 9.5% of the background remains. This is a good sign because it shows the cuts are much harder on the background than on the signal.

The usual sensitivity numbers also improve clearly. The value of `S/B` increases from 0.000777 to 0.005674, and `S/sqrt(B)` increases from 1.81 to 4.06. So even though some signal is lost, the overall separation becomes much better after the selection.

This is really the main numerical check for the cut-based part of the analysis. The project asks for a weighted before-and-after comparison, and these results show that the chosen cuts are reasonable: they keep a useful amount of signal while removing most of the background.

## 2.4 Weighted mass distributions before and after cuts

In [ ]:
# Weighted mass distributions before and after cuts

# I use the same mass bins for both plots so the comparison stays fair
mass_bins_gev = np.linspace(500, 1800, 61)

# Make side-by-side plots to compare the mass spectrum before and after the cuts
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=False)

# Plot the weighted background spectrum before cuts
axes[0].hist(
    background_before_cuts["reco_zv_mass_gev"],
    bins=mass_bins_gev,
    weights=background_before_cuts["FullEventWeight"],
    histtype="stepfilled",
    alpha=0.55,
    label="Combined background",
)

# Overlay the weighted signal spectrum before cuts
axes[0].hist(
    signal_before_cuts["reco_zv_mass_gev"],
    bins=mass_bins_gev,
    weights=signal_before_cuts["FullEventWeight"],
    histtype="step",
    linewidth=2.2,
    color="black",
    label="Signal",
)

axes[0].set_title("Before cuts")
axes[0].set_xlabel(r"$m_{\ell\ell J}$ [GeV]")
axes[0].set_ylabel("Weighted events")
axes[0].legend()

# Plot the weighted background spectrum after cuts
axes[1].hist(
    background_after_cuts["reco_zv_mass_gev"],
    bins=mass_bins_gev,
    weights=background_after_cuts["FullEventWeight"],
    histtype="stepfilled",
    alpha=0.55,
    label="Combined background",
)

# Overlay the weighted signal spectrum after cuts
axes[1].hist(
    signal_after_cuts["reco_zv_mass_gev"],
    bins=mass_bins_gev,
    weights=signal_after_cuts["FullEventWeight"],
    histtype="step",
    linewidth=2.2,
    color="black",
    label="Signal",
)

axes[1].set_title("After cuts")
axes[1].set_xlabel(r"$m_{\ell\ell J}$ [GeV]")
axes[1].set_ylabel("Weighted events")
axes[1].legend()

plt.tight_layout()
plt.show()

## 2.5 Mass spectrum before/after cuts

In [ ]:
# Stacked background after cuts

plt.figure(figsize=(8, 5))

plt.hist(
    [
        diboson_after_cuts["reco_zv_mass_gev"],
        zjets_after_cuts["reco_zv_mass_gev"],
        top_after_cuts["reco_zv_mass_gev"],
    ],
    bins=mass_bins_gev,
    weights=[
        diboson_after_cuts["FullEventWeight"],
        zjets_after_cuts["FullEventWeight"],
        top_after_cuts["FullEventWeight"],
    ],
    stacked=True,
    alpha=0.75,
    label=["Diboson", "Z+jets", "Top"],
)

plt.hist(
    signal_after_cuts["reco_zv_mass_gev"],
    bins=mass_bins_gev,
    weights=signal_after_cuts["FullEventWeight"],
    histtype="step",
    linewidth=2.3,
    color="black",
    label="Signal",
)

plt.xlabel(r"$m_{\ell\ell J}$ [GeV]")
plt.ylabel("Weighted events")
plt.title("Mass spectrum after cuts")
plt.legend()
plt.tight_layout()
plt.show()


After applying the kinematic cuts, I plot the `reco_zv_mass` spectrum again for the combined signal and background sample. I do not apply any cut on this variable itself, since the project brief clearly says not to do that at this stage.

The reason for the earlier cuts is to improve the visibility of any possible signal in the mass distribution, not to shape the mass variable directly. After the selection, the excess near 1 TeV becomes more visible than it was before the cuts. This suggests that the cuts are helping in the way they should: they reduce background enough to make the signal region clearer, while still leaving a mass spectrum that can be used for the later fit and hypothesis test.

# 3 Fitting and hypothesis testing
## 3.1 Signal-only and background-only fits

For the fitting stage, I model the selected signal spectrum with a Gaussian and the selected background spectrum with a smooth polynomial. I use separate fits first because the project asks for the signal and background shapes to be determined independently before constructing the joint model. I also restrict the fit to subranges of the mass spectrum rather than using the full histogram, because the brief suggests that this is often more stable. The Gaussian is a simple approximation for the signal peak, while the polynomial gives a flexible but still smooth description of the falling background shape. I tried different reasonable choices here and kept the simplest model that still gave a sensible visual description of the selected spectra. The background-only fit is good, while the signal-only fit is visually reasonable but not perfect, so I treat the extracted mean and width as an approximate template rather than as an exact description of the signal shape.

In [ ]:
# I choose a narrower mass range for the fit so I focus on the main region of interest
fit_bins_gev = np.linspace(800, 1500, 36)

# This function makes a weighted histogram and also returns the weighted bin errors
# The bin error is taken from the sum of squared weights
def make_weighted_histogram(dataframe, mass_column, bin_edges):
    counts = np.histogram(
        dataframe[mass_column],
        bins=bin_edges,
        weights=dataframe["FullEventWeight"]
    )[0]

    sum_w2 = np.histogram(
        dataframe[mass_column],
        bins=bin_edges,
        weights=dataframe["FullEventWeight"] ** 2
    )[0]

    errors = np.sqrt(sum_w2)
    centres = 0.5 * (bin_edges[:-1] + bin_edges[1:])
    return centres, counts, errors

# I use a Gaussian for the signal shape
def gaussian(x, amplitude, mean, sigma):
    return amplitude * np.exp(-0.5 * ((x - mean) / sigma) ** 2)

# For the background, I use a simple quadratic polynomial
# This is flexible enough for the smooth background shape in this range
def background_poly(x, p0, p1, p2):
    t = (x - 1000.0) / 300.0
    return p0 + p1 * t + p2 * t**2

# Weighted chi-square:
# observed bin content = sum of weights
# bin variance = sum of squared weights
def weighted_chi2(observed, predicted, errors):
    valid = (errors > 0) & (predicted > 0)
    return np.sum(((observed[valid] - predicted[valid]) / errors[valid]) ** 2)

# Build weighted histograms for signal, background, and the combined selected sample
signal_centres, signal_counts, signal_errors = make_weighted_histogram(
    signal_after_cuts, "reco_zv_mass_gev", fit_bins_gev
)

background_centres, background_counts, background_errors = make_weighted_histogram(
    background_after_cuts, "reco_zv_mass_gev", fit_bins_gev
)

joint_after_cuts = pd.concat([signal_after_cuts, background_after_cuts], ignore_index=True)

joint_centres, joint_counts, joint_errors = make_weighted_histogram(
    joint_after_cuts, "reco_zv_mass_gev", fit_bins_gev
)

# I fit the signal only in a region around the peak
signal_mask = (
    (signal_centres >= 920) &
    (signal_centres <= 1080) &
    (signal_errors > 0)
)

# For the background and joint fits, I use a wider range
background_mask = (
    (background_centres >= 850) &
    (background_centres <= 1450) &
    (background_errors > 0)
)

joint_mask = (
    (joint_centres >= 850) &
    (joint_centres <= 1450) &
    (joint_errors > 0)
)

# First fit the signal-only histogram
# This gives the mean and width that I later fix in the H1 fit
def signal_chi2(params):
    amplitude, mean, sigma = params
    predicted = gaussian(signal_centres[signal_mask], amplitude, mean, sigma)
    return weighted_chi2(signal_counts[signal_mask], predicted, signal_errors[signal_mask])

signal_guess = [signal_counts[signal_mask].max(), 1000.0, 80.0]
signal_bounds = [(0, None), (900, 1100), (20, 200)]

signal_fit = minimize(signal_chi2, signal_guess, bounds=signal_bounds)
signal_amplitude_fit, signal_mean_fit, signal_sigma_fit = signal_fit.x
signal_chi2_min = signal_fit.fun
signal_ndof = signal_mask.sum() - 3

# Next fit the background-only spectrum
def background_chi2(params):
    p0, p1, p2 = params
    predicted = background_poly(background_centres[background_mask], p0, p1, p2)
    if np.any(predicted <= 0):
        return 1e30
    return weighted_chi2(background_counts[background_mask], predicted, background_errors[background_mask])

background_guess = [
    np.median(background_counts[background_mask]),
    -0.5 * np.median(background_counts[background_mask]),
    0.1 * np.median(background_counts[background_mask]),
]

background_fit = minimize(background_chi2, background_guess)
background_p0_fit, background_p1_fit, background_p2_fit = background_fit.x
background_chi2_min = background_fit.fun
background_ndof = background_mask.sum() - 3

# H0 means background only
# Here I fit the combined selected spectrum with no signal term
def h0_chi2(params):
    p0, p1, p2 = params
    predicted = background_poly(joint_centres[joint_mask], p0, p1, p2)
    if np.any(predicted <= 0):
        return 1e30
    return weighted_chi2(joint_counts[joint_mask], predicted, joint_errors[joint_mask])

h0_guess = [background_p0_fit, background_p1_fit, background_p2_fit]
h0_fit = minimize(h0_chi2, h0_guess)
h0_p0_fit, h0_p1_fit, h0_p2_fit = h0_fit.x
h0_chi2_min = h0_fit.fun
h0_ndof = joint_mask.sum() - 3

# H1 means signal plus background
# The signal mean and width are fixed from the signal-only fit
# Only the signal amplitude is free here
def h1_chi2(params):
    signal_amp, p0, p1, p2 = params
    predicted = (
        gaussian(joint_centres[joint_mask], signal_amp, signal_mean_fit, signal_sigma_fit)
        + background_poly(joint_centres[joint_mask], p0, p1, p2)
    )
    if np.any(predicted <= 0):
        return 1e30
    return weighted_chi2(joint_counts[joint_mask], predicted, joint_errors[joint_mask])

h1_guess = [signal_amplitude_fit, background_p0_fit, background_p1_fit, background_p2_fit]
h1_bounds = [(0, None), (None, None), (None, None), (None, None)]

h1_fit = minimize(h1_chi2, h1_guess, bounds=h1_bounds)
h1_signal_amp_fit, h1_p0_fit, h1_p1_fit, h1_p2_fit = h1_fit.x
h1_chi2_min = h1_fit.fun
h1_ndof = joint_mask.sum() - 4

# Compare H0 and H1 using Delta chi-square
# With Wilks' theorem, this can be turned into a p-value and Z-value
delta_chi2 = h0_chi2_min - h1_chi2_min
p_value = chi2.sf(delta_chi2, 1)
z_value = norm.isf(p_value)

# Print all fit results clearly
print("Signal-only fit")
print(f"mean  = {signal_mean_fit:.3f} GeV")
print(f"sigma = {signal_sigma_fit:.3f} GeV")
print(f"chi2 / ndof = {signal_chi2_min:.3f} / {signal_ndof}")

print("\nBackground-only fit")
print(f"p0 = {background_p0_fit:.3f}")
print(f"p1 = {background_p1_fit:.3f}")
print(f"p2 = {background_p2_fit:.3f}")
print(f"chi2 / ndof = {background_chi2_min:.3f} / {background_ndof}")

print("\nH0 fit")
print(f"chi2 / ndof = {h0_chi2_min:.3f} / {h0_ndof}")

print("\nH1 fit")
print(f"signal amplitude = {h1_signal_amp_fit:.3f}")
print(f"chi2 / ndof = {h1_chi2_min:.3f} / {h1_ndof}")

print("\nHypothesis test")
print(f"Delta chi2 = {delta_chi2:.3f}")
print(f"p-value    = {p_value:.6g}")
print(f"Z-value    = {z_value:.3f} sigma")

### H0 vs H1 hypothesis test after cuts

After the cuts, I test two nested models on the selected `reco_zv_mass` spectrum. In the background-only hypothesis, $H_0$, the data are described using only the background shape. In the signal-plus-background hypothesis, $H_1$, I add a Gaussian signal term. The Gaussian mean and width are fixed from the signal-only fit, while the signal normalisation is left free, and the background parameters are still allowed to vary. This is the setup requested in the project brief.

Since the events are weighted, I do not use a simple counting-based $\chi^2$. Instead, I use the modified binned $\chi^2$ form in which the content of each bin is the sum of event weights and the variance is taken as the sum of squared weights in that bin. This makes the test consistent with the weighted nature of the dataset.

For the cut-based selection, the fit gives
$$
\chi^2 / \mathrm{ndof} = 80.664/28
$$
for $H_0$, and
$$
\chi^2 / \mathrm{ndof} = 28.726/27
$$
for $H_1$.
So, once the signal component is included, the fit improves significantly.

The difference between the two fits is
$$
\Delta \chi^2 = 51.938,
$$
which corresponds to a p-value of
$$
5.73 \times 10^{-13}
$$
and a significance of
$$
7.11\,\sigma.
$$

The main point is that, after the cuts, the selected mass spectrum is described much better by a model that includes a peak near $1~\mathrm{TeV}$ than by a background-only model. At the same time, the p-value should be interpreted carefully: it is not the probability that $H_0$ is true. Instead, it is the probability of obtaining data at least this extreme if $H_0$ were true.

## 3.2 Plots of fits 

In [ ]:
# Make smooth x-values so the fitted curves look clean on the plots
x_signal_plot = np.linspace(850, 1150, 400)
y_signal_plot = gaussian(x_signal_plot, signal_amplitude_fit, signal_mean_fit, signal_sigma_fit)

x_background_plot = np.linspace(700, 1600, 500)
y_background_plot = background_poly(x_background_plot, background_p0_fit, background_p1_fit, background_p2_fit)

x_joint_plot = np.linspace(700, 1600, 500)
y_h0_plot = background_poly(x_joint_plot, h0_p0_fit, h0_p1_fit, h0_p2_fit)
y_h1_plot = (
    gaussian(x_joint_plot, h1_signal_amp_fit, signal_mean_fit, signal_sigma_fit)
    + background_poly(x_joint_plot, h1_p0_fit, h1_p1_fit, h1_p2_fit)
)

# Plot the signal fit, background fit, and final H0/H1 comparison side by side
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# First plot: signal-only fit
axes[0].errorbar(
    signal_centres[signal_mask],
    signal_counts[signal_mask],
    yerr=signal_errors[signal_mask],
    fmt="o",
    label="Signal after cuts"
)
axes[0].plot(x_signal_plot, y_signal_plot, linewidth=2, label="Gaussian fit")
axes[0].set_xlabel(r"$m_{\ell\ell J}$ [GeV]")
axes[0].set_ylabel("Weighted events")
axes[0].set_title("Signal-only fit")
axes[0].legend()

# Second plot: background-only fit
axes[1].errorbar(
    background_centres[background_mask],
    background_counts[background_mask],
    yerr=background_errors[background_mask],
    fmt="o",
    label="Background after cuts"
)
axes[1].plot(x_background_plot, y_background_plot, linewidth=2, label="Polynomial fit")
axes[1].set_xlabel(r"$m_{\ell\ell J}$ [GeV]")
axes[1].set_ylabel("Weighted events")
axes[1].set_title("Background-only fit")
axes[1].legend()

# Third plot: compare the joint spectrum with H0 and H1
axes[2].errorbar(
    joint_centres[joint_mask],
    joint_counts[joint_mask],
    yerr=joint_errors[joint_mask],
    fmt="o",
    label="Signal + background"
)
axes[2].plot(x_joint_plot, y_h0_plot, linewidth=2, label=r"$H_0$")
axes[2].plot(x_joint_plot, y_h1_plot, linewidth=2, label=r"$H_1$")
axes[2].set_xlabel(r"$m_{\ell\ell J}$ [GeV]")
axes[2].set_ylabel("Weighted events")
axes[2].set_title("Joint fit")
axes[2].legend()

plt.tight_layout()
plt.show()

# 4. NN without reco_zv_mass

In this section, I train a neural-network classifier using the eight kinematic variables given in the course material. As required, I do not include `reco_zv_mass` or `FullEventWeight` as input features.

The project brief says that there is no single correct choice for the training sample, so I tried a few different options before deciding on a looser preselection. I chose this because the full dataset is strongly dominated by background, while the final cut-based sample is smaller and much more specific. Using a looser preselection gives the network a wider range of events to learn from, while still keeping the training relevant to the physics problem.

To train the model, I build a balanced sample with equal numbers of signal and background events. I then shuffle the events, standardise the input variables, and split the sample into training and test sets. For the classifier, I use `MLPClassifier`. I chose it because it is a straightforward non-linear model that can learn correlations between variables without making the analysis overly complicated.

The test results are reasonable and realistic rather than suspiciously perfect. The model gives an accuracy of 0.7954 and a ROC AUC of 0.8759, with fairly balanced precision and recall for both classes. This suggests that the network has learned useful separation power from the kinematic variables alone, even without using the mass variable directly.

## 4.1 NN Model Training and testing 

In [ ]:
# These are the input variables for the first NN
# I do not include reco_zv_mass here because this section is meant to work without mass
nn_input_features = [
    "lep1_pt",
    "lep2_pt",
    "fatjet_pt",
    "fatjet_eta",
    "fatjet_D2",
    "Zll_mass",
    "Zll_pt",
    "MET",
]

random_seed = 42

# I use a looser selection for training
# The idea is to give the network a broader sample, not only the tightly selected final events
def apply_looser_cuts(df):
    return (
        (df["lep1_pt"] > 150000) &
        (df["lep2_pt"] > 60000) &
        (df["fatjet_pt"] > 320000) &
        (df["fatjet_D2"] < 2.2) &
        (df["Zll_pt"] > 240000) &
        (df["MET"] < 100000)
    )

# Build the signal and background samples for NN training, the dataset is 50/ 50 signal/background balance
signal_training_df = signal_df[apply_looser_cuts(signal_df)].copy()
background_training_df = background_df[apply_looser_cuts(background_df)].copy()

# The full sample is very background-heavy, so I balance it here
# I randomly take the same number of background events as signal events
background_training_df = background_training_df.sample(
    n=len(signal_training_df),
    random_state=random_seed
)

# Combine signal and background into one training dataframe
training_df = pd.concat(
    [signal_training_df, background_training_df],
    ignore_index=True
)

# Shuffle the rows so signal and background are mixed properly before training
training_df = shuffle(training_df, random_state=random_seed).reset_index(drop=True)

# Separate inputs and targets
input_data_nn = training_df[nn_input_features].to_numpy()
target_data_nn = training_df["isSignal"].to_numpy()

# Standardise the inputs because the variables have very different numerical scales
scaler_nn = StandardScaler()
input_data_scaled_nn = scaler_nn.fit_transform(input_data_nn)

# Split into training and test samples
# I keep the class balance the same in both using stratify
train_data_nn, test_data_nn, train_target_nn, test_target_nn = train_test_split(
    input_data_scaled_nn,
    target_data_nn,
    test_size=0.3,
    random_state=random_seed,
    stratify=target_data_nn
)

# Use a simple MLP classifier
# I choose this because it can learn non-linear patterns without making the setup too complicated
nn_model = MLPClassifier(
    hidden_layer_sizes=(40,),
    activation="relu",
    solver="adam",
    alpha=5e-4,
    learning_rate_init=1e-3,
    max_iter=250,
    #early_stopping=True,
    validation_fraction=0.2,
    n_iter_no_change=15,
    random_state=random_seed
)

# Train the network
nn_model.fit(train_data_nn, train_target_nn)

# Get the NN output probabilities for the test sample
test_probabilities_nn = nn_model.predict_proba(test_data_nn)[:, 1]

# Turn probabilities into class predictions using 0.5 as the threshold
test_predictions_nn = (test_probabilities_nn > 0.5).astype(int)

# Calculate the main performance metrics
test_accuracy_nn = accuracy_score(test_target_nn, test_predictions_nn)
test_auc_nn = roc_auc_score(test_target_nn, test_probabilities_nn)
confusion_nn = confusion_matrix(test_target_nn, test_predictions_nn)

# Print a short summary of the training sample
print("Training sample")
print(f"Signal rows: {len(signal_training_df):,}")
print(f"Background rows: {len(background_training_df):,}")
print(f"Total rows: {len(training_df):,}")

# Print the main test results
print("\nTest results")
print(f"Accuracy: {test_accuracy_nn:.4f}")
print(f"ROC AUC : {test_auc_nn:.4f}")
print("\nConfusion matrix")
print(confusion_nn)
print("\nClassification report")
print(classification_report(test_target_nn, test_predictions_nn, digits=4))

### NN performance

The classifier gives an accuracy of 0.7954 and a ROC AUC of 0.8759 on the test sample. These results are good enough to show that the network has learned useful separation between signal and background using only the kinematic variables. The precision and recall are also fairly balanced for the two classes, which suggests that the model is not strongly biased toward only signal or only background. So, even without using reco_zv_mass, the NN already captures useful information for improving the search selection.

## 4.2 ROC curve and Confusion Matrix

In [ ]:
# First I plot the confusion matrix and ROC curve
# These help me check how well the NN is separating signal and background on the test sample
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].imshow(confusion_nn)
axes[0].set_title("Confusion matrix")
axes[0].set_xticks([0, 1])
axes[0].set_yticks([0, 1])
axes[0].set_xticklabels(["Background", "Signal"])
axes[0].set_yticklabels(["Background", "Signal"])
axes[0].set_xlabel("Predicted")
axes[0].set_ylabel("True")

# Write the event counts inside the confusion matrix cells
for i in range(2):
    for j in range(2):
        axes[0].text(j, i, confusion_nn[i, j], ha="center", va="center")

# Plot the ROC curve using the NN output probabilities
fpr_nn, tpr_nn, thresholds_nn = roc_curve(test_target_nn, test_probabilities_nn)
axes[1].plot(fpr_nn, tpr_nn, linewidth=2, label=f"AUC = {test_auc_nn:.3f}")
axes[1].plot([0, 1], [0, 1], linestyle="--")
axes[1].set_xlabel("False positive rate")
axes[1].set_ylabel("True positive rate")
axes[1].set_title("ROC curve")
axes[1].legend()

plt.tight_layout()
plt.show()

# Now I apply the trained NN to the cut-based selected signal and background samples
signal_application_df_nn = signal_after_cuts.copy()
background_application_df_nn = background_after_cuts.copy()

# Make sure the mass in GeV is available for plotting later
if "reco_zv_mass_gev" not in signal_application_df_nn.columns:
    signal_application_df_nn["reco_zv_mass_gev"] = signal_application_df_nn["reco_zv_mass"] / 1000.0
if "reco_zv_mass_gev" not in background_application_df_nn.columns:
    background_application_df_nn["reco_zv_mass_gev"] = background_application_df_nn["reco_zv_mass"] / 1000.0

# Get the NN score for every event in the signal sample
signal_application_df_nn["nn_score"] = nn_model.predict_proba(
    scaler_nn.transform(signal_application_df_nn[nn_input_features].to_numpy())
)[:, 1]

# Get the NN score for every event in the background sample
background_application_df_nn["nn_score"] = nn_model.predict_proba(
    scaler_nn.transform(background_application_df_nn[nn_input_features].to_numpy())
)[:, 1]

# Keep only events with NN score above 0.5, following the project requirement
signal_nn_selected = signal_application_df_nn[signal_application_df_nn["nn_score"] > 0.5].copy()
background_nn_selected = background_application_df_nn[background_application_df_nn["nn_score"] > 0.5].copy()

# Combine the NN-selected signal and background for later use
joint_nn_selected = pd.concat(
    [signal_nn_selected, background_nn_selected],
    ignore_index=True
)

# Weighted yield is what matters here, not just the number of rows
def weighted_yield(df):
    return df["FullEventWeight"].sum()

# Compare yields before and after the NN selection
signal_yield_cut_nn = weighted_yield(signal_after_cuts)
background_yield_cut_nn = weighted_yield(background_after_cuts)

signal_yield_nn = weighted_yield(signal_nn_selected)
background_yield_nn = weighted_yield(background_nn_selected)

print("After NN selection")
print(f"Signal rows kept: {len(signal_nn_selected):,}")
print(f"Background rows kept: {len(background_nn_selected):,}")

print("\nWeighted yields after cut-based selection")
print(f"Signal: {signal_yield_cut_nn:,.3f}")
print(f"Background: {background_yield_cut_nn:,.3f}")

print("\nWeighted yields after cut-based + NN")
print(f"Signal: {signal_yield_nn:,.3f}")
print(f"Background: {background_yield_nn:,.3f}")

# Print a simple comparison to see whether the NN improves the selection
print("\nComparison")
print(f"Signal efficiency after NN     = {signal_yield_nn / signal_yield_cut_nn:.4f}")
print(f"Background efficiency after NN = {background_yield_nn / background_yield_cut_nn:.4f}")
print(f"S/B after cuts                = {signal_yield_cut_nn / background_yield_cut_nn:.6f}")
print(f"S/B after cuts + NN           = {signal_yield_nn / background_yield_nn:.6f}")
print(f"S/sqrt(B) after cuts          = {signal_yield_cut_nn / np.sqrt(background_yield_cut_nn):.6f}")
print(f"S/sqrt(B) after cuts + NN     = {signal_yield_nn / np.sqrt(background_yield_nn):.6f}")

# Finally, plot the mass spectrum after the NN selection
# This shows whether the peak becomes cleaner after using the classifier
mass_bins_gev_nn = np.linspace(850, 1450, 31)

plt.figure(figsize=(8, 5))

plt.hist(
    background_nn_selected["reco_zv_mass_gev"],
    bins=mass_bins_gev_nn,
    weights=background_nn_selected["FullEventWeight"],
    histtype="stepfilled",
    alpha=0.6,
    label="Background after NN"
)

plt.hist(
    signal_nn_selected["reco_zv_mass_gev"],
    bins=mass_bins_gev_nn,
    weights=signal_nn_selected["FullEventWeight"],
    histtype="step",
    linewidth=2.2,
    color="black",
    label="Signal after NN"
)

plt.xlabel(r"$m_{\ell\ell J}$ [GeV]")
plt.ylabel("Weighted events")
plt.title("Cleaned mass spectrum after NN selection")
plt.legend()
plt.tight_layout()
plt.show()

### NN-selected sample and cleaned mass spectrum

After training the classifier, I apply it to the selected sample and keep only the events with a predicted signal probability greater than 0.5, following the project instructions. This gives a much cleaner mass spectrum.

In terms of weighted yields, the signal changes from 2901.36 after the cut-based selection to 2196.85 after the neural-network selection. The background, however, is reduced much more strongly, falling from 511349.75 to 73848.73. This shows that the network removes a large fraction of the remaining background while still keeping a useful part of the signal.

Because of that, the simple sensitivity measures improve clearly. The value of $S/B$ increases from 0.005674 to 0.029748, and $S/\sqrt{B}$ increases from 4.06 to 8.08. So the neural network gives a substantial improvement over the cut-based result alone.

This is also visible in the mass plot itself. After the NN selection, the excess near $1~\mathrm{TeV}$ becomes more pronounced, which suggests that the classifier is doing what it is supposed to do: it improves the signal–background separation beyond what was achieved using only the square cuts.

## 4.3 $H_0$ vs $H_1$ after NN

In [ ]:
# Here I repeat the fit after the NN selection
# I use a slightly narrower mass range because now I only want to study the cleaned spectrum
fit_bins_gev_nn = np.linspace(900, 1400, 26)

# This makes the weighted histogram and its bin errors
# The errors come from the sum of squared weights
def make_weighted_histogram_nn(dataframe, mass_column, bin_edges):
    counts = np.histogram(
        dataframe[mass_column],
        bins=bin_edges,
        weights=dataframe["FullEventWeight"]
    )[0]

    sum_w2 = np.histogram(
        dataframe[mass_column],
        bins=bin_edges,
        weights=dataframe["FullEventWeight"] ** 2
    )[0]

    errors = np.sqrt(sum_w2)
    centres = 0.5 * (bin_edges[:-1] + bin_edges[1:])
    return centres, counts, errors

# Signal shape after NN: still modelled with a Gaussian peak
def gaussian_shape_nn(x, amplitude, mean, sigma):
    return amplitude * np.exp(-0.5 * ((x - mean) / sigma) ** 2)

# Background shape after NN
# I use an exponential form here because the NN-selected background is more distorted
def background_shape_nn(x, log_norm, slope, curvature):
    t = (x - 1000.0) / 300.0
    return np.exp(log_norm + slope * t + curvature * t**2)

# Weighted chi-square for the binned fit
def weighted_chi2_nn(observed, predicted, errors):
    valid = (errors > 0) & (predicted > 0)
    return np.sum(((observed[valid] - predicted[valid]) / errors[valid]) ** 2)

# Build histograms for the NN-selected signal, background, and total sample
signal_centres_nn, signal_counts_nn, signal_errors_nn = make_weighted_histogram_nn(
    signal_nn_selected, "reco_zv_mass_gev", fit_bins_gev_nn
)

background_centres_nn, background_counts_nn, background_errors_nn = make_weighted_histogram_nn(
    background_nn_selected, "reco_zv_mass_gev", fit_bins_gev_nn
)

joint_centres_nn, joint_counts_nn, joint_errors_nn = make_weighted_histogram_nn(
    joint_nn_selected, "reco_zv_mass_gev", fit_bins_gev_nn
)

# Fit the signal peak in a tighter region around 1 TeV
signal_fit_mask_nn = (
    (signal_centres_nn >= 940) &
    (signal_centres_nn <= 1060) &
    (signal_errors_nn > 0)
)

# Use a wider region for the background-only and joint fits
background_fit_mask_nn = (
    (background_centres_nn >= 900) &
    (background_centres_nn <= 1400 ) &
    (background_errors_nn > 0)
)

joint_fit_mask_nn = (
    (joint_centres_nn >= 900) &
    (joint_centres_nn <= 1400) &
    (joint_errors_nn > 0)
)

# First fit the signal-only spectrum after NN
# This gives the peak mean and width used later in H1
def signal_chi2_nn(params):
    amplitude, mean, sigma = params
    predicted = gaussian_shape_nn(signal_centres_nn[signal_fit_mask_nn], amplitude, mean, sigma)
    return weighted_chi2_nn(signal_counts_nn[signal_fit_mask_nn], predicted, signal_errors_nn[signal_fit_mask_nn])

signal_guess_nn = [signal_counts_nn[signal_fit_mask_nn].max(), 1000.0, 80.0]
signal_bounds_nn = [(0, None), (900, 1100), (20, 200)]

signal_fit_nn = minimize(signal_chi2_nn, signal_guess_nn, bounds=signal_bounds_nn)
signal_amplitude_fit_nn, signal_mean_fit_nn, signal_sigma_fit_nn = signal_fit_nn.x
signal_chi2_min_nn = signal_fit_nn.fun
signal_ndof_nn = signal_fit_mask_nn.sum() - 3

# Fit the NN-selected background on its own
def background_chi2_nn(params):
    log_norm, slope, curvature = params
    predicted = background_shape_nn(background_centres_nn[background_fit_mask_nn], log_norm, slope, curvature)
    return weighted_chi2_nn(background_counts_nn[background_fit_mask_nn], predicted, background_errors_nn[background_fit_mask_nn])

background_guess_nn = [
    np.log(np.maximum(np.median(background_counts_nn[background_fit_mask_nn]), 1.0)),
    -0.3,
    -0.2,
]

background_fit_nn = minimize(background_chi2_nn, background_guess_nn)
background_log_norm_fit_nn, background_slope_fit_nn, background_curvature_fit_nn = background_fit_nn.x
background_chi2_min_nn = background_fit_nn.fun
background_ndof_nn = background_fit_mask_nn.sum() - 3

# H0 = background only
# Here I test whether the full NN-selected spectrum can be explained without a signal peak
def joint_h0_chi2_nn(params):
    log_norm, slope, curvature = params
    predicted = background_shape_nn(joint_centres_nn[joint_fit_mask_nn], log_norm, slope, curvature)
    return weighted_chi2_nn(joint_counts_nn[joint_fit_mask_nn], predicted, joint_errors_nn[joint_fit_mask_nn])

h0_guess_nn = [background_log_norm_fit_nn, background_slope_fit_nn, background_curvature_fit_nn]
h0_fit_nn = minimize(joint_h0_chi2_nn, h0_guess_nn)
h0_log_norm_fit_nn, h0_slope_fit_nn, h0_curvature_fit_nn = h0_fit_nn.x
h0_chi2_min_nn = h0_fit_nn.fun
h0_ndof_nn = joint_fit_mask_nn.sum() - 3

# H1 = signal plus background
# The signal mean and width stay fixed, while the signal size is allowed to float
def joint_h1_chi2_nn(params):
    signal_amp, log_norm, slope, curvature = params
    predicted = (
        gaussian_shape_nn(joint_centres_nn[joint_fit_mask_nn], signal_amp, signal_mean_fit_nn, signal_sigma_fit_nn)
        + background_shape_nn(joint_centres_nn[joint_fit_mask_nn], log_norm, slope, curvature)
    )
    return weighted_chi2_nn(joint_counts_nn[joint_fit_mask_nn], predicted, joint_errors_nn[joint_fit_mask_nn])

h1_guess_nn = [signal_amplitude_fit_nn, background_log_norm_fit_nn, background_slope_fit_nn, background_curvature_fit_nn]
h1_bounds_nn = [(0, None), (None, None), (None, None), (None, None)]

h1_fit_nn = minimize(joint_h1_chi2_nn, h1_guess_nn, bounds=h1_bounds_nn)
h1_signal_amp_fit_nn, h1_log_norm_fit_nn, h1_slope_fit_nn, h1_curvature_fit_nn = h1_fit_nn.x
h1_chi2_min_nn = h1_fit_nn.fun
h1_ndof_nn = joint_fit_mask_nn.sum() - 4

# Compare H0 and H1 with Delta chi-square
# This gives the p-value and significance using Wilks' theorem
delta_chi2_nn = h0_chi2_min_nn - h1_chi2_min_nn
p_value_nn = chi2.sf(delta_chi2_nn, 1)
z_value_nn = norm.isf(p_value_nn)

# Print all fit results clearly
print("Signal-only fit after NN")
print(f"Amplitude: {signal_amplitude_fit_nn:.3f}")
print(f"Mean: {signal_mean_fit_nn:.3f} GeV")
print(f"Sigma: {signal_sigma_fit_nn:.3f} GeV")
print(f"chi2 / ndof = {signal_chi2_min_nn:.3f} / {signal_ndof_nn}")

print("\nBackground-only fit after NN")
print(f"log_norm: {background_log_norm_fit_nn:.3f}")
print(f"slope: {background_slope_fit_nn:.3f}")
print(f"curvature: {background_curvature_fit_nn:.3f}")
print(f"chi2 / ndof = {background_chi2_min_nn:.3f} / {background_ndof_nn}")

print("\nH0 fit after NN")
print(f"chi2 / ndof = {h0_chi2_min_nn:.3f} / {h0_ndof_nn}")

print("\nH1 fit after NN")
print(f"Signal amplitude: {h1_signal_amp_fit_nn:.3f}")
print(f"chi2 / ndof = {h1_chi2_min_nn:.3f} / {h1_ndof_nn}")

print("\nHypothesis test after NN")
print(f"Delta chi2 = {delta_chi2_nn:.3f}")
print(f"p-value    = {p_value_nn:.6g}")
print(f"Z-value    = {z_value_nn:.3f} sigma")



I then repeat the same $H_0$ and $H_1$ fitting procedure on the NN-selected mass spectrum. Numerically, the result looks much stronger than in the cut-based case. I find
$$
\Delta \chi^2 = 297.816,
$$
which corresponds to a p-value of
$$
9.85 \times 10^{-67}
$$
and a significance of
$$
17.22\,\sigma.
$$

However, I do not treat this as the most reliable final significance of the analysis. The reason is that, after the NN selection, the fit quality becomes poor. Both the background-only fit and the signal-plus-background fit give very large values of
$$
\chi^2/\mathrm{ndof},
$$
which means that the chosen model is no longer describing the selected mass spectrum well.

This is an important point, because it matches the warning in the project brief: once the selection becomes too dependent on variables that are strongly correlated with the mass shape, the final spectrum can become distorted and harder to model reliably. So even though the NN makes the peak near $1~\mathrm{TeV}$ much more visible, the fitted shape is less trustworthy.

My honest conclusion is that the neural network clearly improves the classification power, but the very large Wilks significance after the NN cut should not be taken at face value. In other words, the NN helps to separate signal from background, but it also makes the final mass spectrum harder to describe with the simple fit model used here.

# 4.4 Plots of Fits after NN 

In [ ]:
# Make smooth x-values so the fitted curves look clean on the final plots
x_signal_plot_nn = np.linspace(920, 1080, 400)
y_signal_plot_nn = gaussian_shape_nn(
    x_signal_plot_nn,
    signal_amplitude_fit_nn,
    signal_mean_fit_nn,
    signal_sigma_fit_nn
)

x_background_plot_nn = np.linspace(850, 1450, 500)
y_background_plot_nn = background_shape_nn(
    x_background_plot_nn,
    background_log_norm_fit_nn,
    background_slope_fit_nn,
    background_curvature_fit_nn
)

x_joint_plot_nn = np.linspace(850, 1450, 500)
y_h0_plot_nn = background_shape_nn(
    x_joint_plot_nn,
    h0_log_norm_fit_nn,
    h0_slope_fit_nn,
    h0_curvature_fit_nn
)

# H1 is the sum of the fitted signal peak and the fitted background shape
y_h1_plot_nn = (
    gaussian_shape_nn(x_joint_plot_nn, h1_signal_amp_fit_nn, signal_mean_fit_nn, signal_sigma_fit_nn)
    + background_shape_nn(x_joint_plot_nn, h1_log_norm_fit_nn, h1_slope_fit_nn, h1_curvature_fit_nn)
)

# Plot the signal fit, background fit, and H0/H1 comparison after the NN selection
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# First plot: signal-only fit after NN
axes[0].errorbar(
    signal_centres_nn[signal_fit_mask_nn],
    signal_counts_nn[signal_fit_mask_nn],
    yerr=signal_errors_nn[signal_fit_mask_nn],
    fmt="o",
    label="Signal after NN"
)
axes[0].plot(x_signal_plot_nn, y_signal_plot_nn, linewidth=2, label="Gaussian fit")
axes[0].set_xlabel(r"$m_{\ell\ell J}$ [GeV]")
axes[0].set_ylabel("Weighted events")
axes[0].set_title("Signal-only fit after NN")
axes[0].legend()

# Second plot: background-only fit after NN
axes[1].errorbar(
    background_centres_nn[background_fit_mask_nn],
    background_counts_nn[background_fit_mask_nn],
    yerr=background_errors_nn[background_fit_mask_nn],
    fmt="o",
    label="Background after NN"
)
axes[1].plot(x_background_plot_nn, y_background_plot_nn, linewidth=2, label="Background fit")
axes[1].set_xlabel(r"$m_{\ell\ell J}$ [GeV]")
axes[1].set_ylabel("Weighted events")
axes[1].set_title("Background-only fit after NN")
axes[1].legend()

# Third plot: compare the NN-selected spectrum with H0 and H1
axes[2].errorbar(
    joint_centres_nn[joint_fit_mask_nn],
    joint_counts_nn[joint_fit_mask_nn],
    yerr=joint_errors_nn[joint_fit_mask_nn],
    fmt="o",
    label="Signal + background after NN"
)
axes[2].plot(x_joint_plot_nn, y_h0_plot_nn, linewidth=2, label=r"$H_0$")
axes[2].plot(x_joint_plot_nn, y_h1_plot_nn, linewidth=2, label=r"$H_1$")
axes[2].set_xlabel(r"$m_{\ell\ell J}$ [GeV]")
axes[2].set_ylabel("Weighted events")
axes[2].set_title("Joint fit after NN")
axes[2].legend()

plt.tight_layout()
plt.show()

#### Fit after NN selection

After applying the NN cut, the signal peak around $1~\mathrm{TeV}$ becomes clearer. The signal-only fit is still reasonably good, and the Gaussian describes the main peak quite well. However, the background-only fit is not very good, because after the NN selection the background shape is no longer a simple smooth falling distribution. In other words, the NN improves the separation, but it also changes the shape of the background spectrum.

In the joint fit, $H_1$ gives a better description than $H_0$, especially near the signal region, so this does show that the NN helps to make the signal more visible. At the same time, the overall fit quality remains poor, so I do not treat the very large significance as the most reliable final result.

So my main interpretation of this section is that the NN clearly improves the signal–background separation, but the cut-based fit still gives the more stable and trustworthy statistical result.

# 5 NN including the reco_zv_mass


## 5.1 Define features and prepare training sample

In [ ]:
# Here I use the same inputs as Section 5, but now I also include reco_zv_mass
# The idea is to see how much the NN improves once mass is given directly
features_with_mass = [
    "lep1_pt",
    "lep2_pt",
    "fatjet_pt",
    "fatjet_eta",
    "fatjet_D2",
    "Zll_mass",
    "Zll_pt",
    "MET",
    "reco_zv_mass"
]

# Build a balanced signal and background sample
signal_train_6 = signal_df[features_with_mass].copy()
signal_train_6["isSignal"] = 1

background_train_6 = background_df[features_with_mass].copy()
background_train_6["isSignal"] = 0

# Keep the same number of background events as signal events
background_train_6 = background_train_6.sample(
    n=len(signal_train_6),
    random_state=42
)

# Combine and shuffle the full training sample
train_df_6 = pd.concat([signal_train_6, background_train_6], ignore_index=True)
train_df_6 = train_df_6.sample(frac=1, random_state=42).reset_index(drop=True)

X6 = train_df_6[features_with_mass]
y6 = train_df_6["isSignal"]

print("Section 6 training sample")
print(f"Signal rows: {len(signal_train_6):,}")
print(f"Background rows: {len(background_train_6):,}")
print(f"Total rows: {len(train_df_6):,}")

## 5.2 Train and evaluate the NN

In [ ]:
# Split the data so I can train on one part and test on unseen events
X6_train, X6_test, y6_train, y6_test = train_test_split(
    X6, y6, test_size=0.30, random_state=42, stratify=y6
)

# Scale the inputs because the variables are on different ranges
scaler6 = StandardScaler()
X6_train_scaled = scaler6.fit_transform(X6_train)
X6_test_scaled = scaler6.transform(X6_test)

# Train the NN with reco_zv_mass included
model6 = MLPClassifier(
    hidden_layer_sizes=(64, 32),
    activation="relu",
    solver="adam",
    max_iter=250,
    early_stopping=True,
    validation_fraction=0.15,
    n_iter_no_change=15,
    random_state=42
)

model6.fit(X6_train_scaled, y6_train)

# Test the model performance
y6_prob = model6.predict_proba(X6_test_scaled)[:, 1]
y6_pred = (y6_prob >= 0.5).astype(int)

print("Section 6 results")
print("Accuracy :", round(accuracy_score(y6_test, y6_pred), 4))
print("ROC AUC  :", round(roc_auc_score(y6_test, y6_prob), 4))
print("\nConfusion matrix")
print(confusion_matrix(y6_test, y6_pred))
print("\nClassification report")
print(classification_report(y6_test, y6_pred, digits=4))

## 5.3 Apply the model to the cut-selected sample

In [ ]:
# Now apply the trained model to the cut-selected signal and background samples
signal_sec6 = signal_after_cuts.copy()
background_sec6 = background_after_cuts.copy()

signal_sec6_scaled = scaler6.transform(signal_sec6[features_with_mass])
background_sec6_scaled = scaler6.transform(background_sec6[features_with_mass])

signal_sec6["nn_score_mass"] = model6.predict_proba(signal_sec6_scaled)[:, 1]
background_sec6["nn_score_mass"] = model6.predict_proba(background_sec6_scaled)[:, 1]

# Keep events with score above 0.5
signal_sec6_selected = signal_sec6[signal_sec6["nn_score_mass"] >= 0.5].copy()
background_sec6_selected = background_sec6[background_sec6["nn_score_mass"] >= 0.5].copy()

print("After Section 6 NN selection")
print(f"Selected signal entries    : {len(signal_sec6_selected):,}")
print(f"Selected background entries: {len(background_sec6_selected):,}")
print(f"Weighted signal yield      : {signal_sec6_selected['FullEventWeight'].sum():,.3f}")
print(f"Weighted background yield  : {background_sec6_selected['FullEventWeight'].sum():,.3f}")

## 5.4 Plot Section 5 result and compare with Section 4

In [ ]:
# First show the mass spectrum after the Section 6 NN selection
plt.figure(figsize=(10, 6))

plt.hist(
    background_sec6_selected["reco_zv_mass"] / 1000.0,
    bins=40,
    range=(500, 1500),
    weights=background_sec6_selected["FullEventWeight"],
    histtype="step",
    linewidth=2,
    label="Background (with mass in NN)"
)

plt.hist(
    signal_sec6_selected["reco_zv_mass"] / 1000.0,
    bins=40,
    range=(500, 1500),
    weights=signal_sec6_selected["FullEventWeight"],
    histtype="step",
    linewidth=2,
    label="Signal (with mass in NN)"
)

plt.xlabel("reco_zv_mass [GeV]")
plt.ylabel("Weighted events")
plt.title("Section 6: mass spectrum after NN selection with reco_zv_mass included")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

# Then compare Section 5 and Section 6 side by side
fig, ax = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

ax[0].hist(
    background_nn_selected["reco_zv_mass"] / 1000.0,
    bins=40,
    range=(500, 1500),
    weights=background_nn_selected["FullEventWeight"],
    histtype="step",
    linewidth=2,
    label="Background"
)

ax[0].hist(
    signal_nn_selected["reco_zv_mass"] / 1000.0,
    bins=40,
    range=(500, 1500),
    weights=signal_nn_selected["FullEventWeight"],
    histtype="step",
    linewidth=2,
    label="Signal"
)

ax[0].set_title("Section 5: NN without mass")
ax[0].set_xlabel("reco_zv_mass [GeV]")
ax[0].set_ylabel("Weighted events")
ax[0].legend()
ax[0].grid(alpha=0.3)

ax[1].hist(
    background_sec6_selected["reco_zv_mass"] / 1000.0,
    bins=40,
    range=(500, 1500),
    weights=background_sec6_selected["FullEventWeight"],
    histtype="step",
    linewidth=2,
    label="Background"
)

ax[1].hist(
    signal_sec6_selected["reco_zv_mass"] / 1000.0,
    bins=40,
    range=(500, 1500),
    weights=signal_sec6_selected["FullEventWeight"],
    histtype="step",
    linewidth=2,
    label="Signal"
)

ax[1].set_title("Section 6: NN with mass")
ax[1].set_xlabel("reco_zv_mass [GeV]")
ax[1].legend()
ax[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## NN with mass included

In this section, I train a second classifier, but this time I include `reco_zv_mass` as one of the input features. This is allowed in the project brief, which asks for a second study where the mass information is incorporated into the training in some way. In this section, I do not repeat the mass-spectrum fits, again following the instructions.

As expected, the classifier performance becomes much stronger once mass is included. The accuracy increases to 0.945 and the ROC AUC rises to 0.983. This improvement is not surprising, because `reco_zv_mass` already contains the main information that separates the signal from the background: the signal is expected to produce a peak near $1~\mathrm{TeV}$.

So I do not interpret this result in the same way as the previous network. In Section 5, the classifier had to learn from the kinematic variables alone, which made it a fairer test of whether the non-mass features carry useful separation power. Here, once the mass variable is given directly to the network, the classification problem becomes much easier. For that reason, this section is less about discovering subtle kinematic structure and more about showing how strongly the classifier performance changes when the main search variable is included explicitly.

# 6. Comparison of Section 5 and Section 6

In [ ]:

# Comparison of Section 5 and Section 6

def weighted_yield(df):
    """Return the weighted event yield."""
    return df["FullEventWeight"].sum()

def safe_ratio(num, den):
    """Return num / den, protecting against division by zero."""
    return num / den if den > 0 else np.nan

# Section 5: NN without reco_zv_mass
signal_yield_sec5 = weighted_yield(signal_nn_selected)
background_yield_sec5 = weighted_yield(background_nn_selected)

sb_sec5 = safe_ratio(signal_yield_sec5, background_yield_sec5)
ssqrtb_sec5 = safe_ratio(signal_yield_sec5, np.sqrt(background_yield_sec5))

# Section 6: NN with reco_zv_mass
signal_yield_sec6 = weighted_yield(signal_sec6_selected)
background_yield_sec6 = weighted_yield(background_sec6_selected)

sb_sec6 = safe_ratio(signal_yield_sec6, background_yield_sec6)
ssqrtb_sec6 = safe_ratio(signal_yield_sec6, np.sqrt(background_yield_sec6))

# Put the comparison into a small table
comparison_df = pd.DataFrame({
    "Section": ["Section 5: NN without mass", "Section 6: NN with mass"],
    "Weighted signal yield": [signal_yield_sec5, signal_yield_sec6],
    "Weighted background yield": [background_yield_sec5, background_yield_sec6],
    "S/B": [sb_sec5, sb_sec6],
    "S/sqrt(B)": [ssqrtb_sec5, ssqrtb_sec6],
})

print("Comparison of Section 5 and Section 6")
display(comparison_df.round(4))

print(f"Section 5: S/B = {sb_sec5:.4f}, S/sqrt(B) = {ssqrtb_sec5:.4f}")
print(f"Section 6: S/B = {sb_sec6:.4f}, S/sqrt(B) = {ssqrtb_sec6:.4f}")

# Short interpretation
if sb_sec5 > sb_sec6:
    print("\nSection 5 gives the cleaner signal region in terms of S/B.")
else:
    print("\nSection 6 gives the cleaner signal region in terms of S/B.")

if ssqrtb_sec5 > ssqrtb_sec6:
    print("Section 5 also gives the better simple sensitivity in terms of S/sqrt(B).")
else:
    print("Section 6 gives the better simple sensitivity in terms of S/sqrt(B).")

### Comparison of Section 5 and Section 6

Section 6 gives much better classifier scores, but it does not give a better search region. In Section 5, the NN without `reco_zv_mass` gives a higher \(S/B\) and a higher \(S/\sqrt{B}\) than Section 6. After adding `reco_zv_mass`, the network keeps slightly more signal, but it also keeps much more background in the same mass region. So the classifier looks stronger in terms of accuracy and ROC AUC, but the simple search sensitivity becomes worse. I therefore treat Section 6 as a study of classifier behaviour rather than as the best final search strategy.

# 7. Final conclusion

In this notebook, I studied a weighted search for a heavy signal near $1~\mathrm{TeV}$ using both a cut-based method and a neural-network method. I first compared the main kinematic distributions and then chose simple square cuts by eye. I tried several different cut values before deciding on the final selection. My aim was not only to reduce the background as much as possible, but to find a sensible balance where the signal became more visible without making the mass spectrum look too artificial.

The cut-based selection already improved the analysis in a clear way. It increased both $S/B$ and $S/\sqrt{B}$, and the selected mass spectrum showed a more visible excess near $1~\mathrm{TeV}$. When I performed the $H_0$ versus $H_1$ fit after the cuts, the signal-plus-background hypothesis described the data much better than the background-only hypothesis. This gave a strong result and, in my view, it is the most reliable statistical result in the notebook.

I then trained a neural network without using `reco_zv_mass`, so that the model had to learn from the event kinematics rather than from the mass peak directly. This classifier gave reasonable performance and improved the signal–background separation further. After applying the NN cut, the signal region became cleaner and the simple sensitivity measures improved again. However, when I repeated the mass-spectrum fit after the NN selection, the fit quality became poor, especially for the background description. This means that the NN helped to separate signal from background, but it also changed the selected mass shape enough that the very large fitted significance should be treated with caution.

Finally, I trained a second neural network including `reco_zv_mass`. As expected, the classifier performance became much stronger, because the model was now using the main signal-sensitive variable directly. But this did not give a better final search region. Even though more signal was kept, more background was also selected in the same mass region, so both $S/B$ and $S/\sqrt{B}$ became worse than in the NN study without mass. For that reason, I treat this last part mainly as a comparison study showing how including the mass variable changes the behaviour of the classifier.

Overall, the main lesson from this project is that better ML performance does not always mean better physics sensitivity. In my case, the most defensible final result is the cut-based $H_0$ versus $H_1$ fit, while the neural-network sections are useful because they show both the benefit and the limitation of ML-based event selection in a search analysis.

# 8. Acknowledgement 
I get some knowledge from reading out the last semester hypothesis test and p value pdf and workshops, I also use the university provided LLM to improve my markdwon quality and debuging my code and improve the code. 